# gambit_teach: the player model and the teacher policy

This package is the original-design core of Gambit (README "Why Gambit"): estimate a player's
**current** level from their **habits** instead of their results, and choose moves that target
those habits while keeping the game close.

## Constructive smurfing

"Smurfing" normally means a strong player hiding behind a low rating to beat weaker opponents.
Gambit uses it in reverse, for the player's benefit: it deliberately plays below full strength,
aims its moves at the player's weak points (recurring mistakes, missed tactics), and reveals more of
its real ability as the player improves. Design goals:

* keep games close, so players stay engaged;
* punish habits, not playstyles (safe, aggressive and passionate styles are all valid);
* build confidence in decision making, pattern recognition and handling pressure.

A returning, "rusty" player keeps their intuition but plays sloppier; rating systems punish them
for that. Gambit matches their current habits instead of an old rating.

## Pipeline

```
PGN history / robot logs --habits.ipynb--> MoveAnalysis per move --> HabitProfile
                                         \--player_model.ipynb--> PlayerModel (Elo +- CI, weaknesses, confidence, OOD)
position + PlayerModel --move_selection.ipynb--> TeacherPolicy.choose_move  (fallback when unsure / OOD)
finished games --metrics.ipynb--> balanced share, eval swings, closeness
stand-in players --ood_testing.ipynb--> does the model fail safely?
```

| Notebook | What | Status |
|---|---|---|
| `habits.ipynb` | CPL, classification, phase, missed forcing moves, HabitProfile | implemented; tactic detectors TODO |
| `player_model.ipynb` | Elo estimate + CI, weaknesses, confidence, OOD score | implemented with placeholder formulas (TODO-calibrate) |
| `move_selection.ipynb` | teacher policy: fallback + band filter + scoring | fallback/scoring implemented; targeting TODO |
| `metrics.ipynb` | balanced-outcomes metric | implemented (definition proposed) |
| `ood_testing.ipynb` | personas + harness | harness implemented; styled personas TODO |
| `disclosure.ipynb` | design notes on disclosing strength adjustment | open question |
| `run_analyze_history.ipynb` | import PGNs (e.g. lichess export) -> player model | run notebook |
| `run_teaching_experiment.ipynb` | teacher vs personas, metrics | run notebook |

Run order: `habits` -> `metrics` -> `player_model` -> `move_selection` -> `ood_testing`, then the
run notebooks. Engine code comes from `../../gambit_game/notebooks/engine.ipynb`; settings from
`gambit_bringup/config/game.yaml` (`teaching:` section).

## Teaching milestones (README)

- [ ] Estimate player habits from game history (`habits`, `player_model`, `run_analyze_history`)
- [ ] Fallback behaviour when the model is uncertain or the player departs from history (`move_selection`)
- [ ] Out-of-distribution testing with stand-in players (`ood_testing`, `run_teaching_experiment`)
- [ ] Rating-targeted play at around 1700 Elo (`gambit_game/run_engine_strength`, `move_selection`)
- [ ] Balanced outcomes: define the metric and a target over N games (`metrics`)
- [ ] Stretch: adapt to players near 2500+ Elo (needs a stronger analyst and calibration there)

## Open questions (README)

* How will player habits be measured, and from how many games? (`teaching.confidence.min_games`)
* Should the robot disclose that it is adjusting its strength? (`disclosure.ipynb`)
* What should Gambit do when the player's play departs sharply from their history? (current
  answer: fall back to solid play at the last estimated level)
* Should Gambit ever play at full strength to avoid losing, or are losses an accepted part of
  teaching? (README claims: losses happen by design)
* What are the minimum and maximum strengths Gambit will play at? (`engine.min_elo` / `max_elo`)